# Market Regime Detection using K-Means Clustering

## Objective

The objective of this notebook is to identify hidden market regimes within Bitcoin price behavior using unsupervised machine learning.

After transforming the original financial features through Principal Component Analysis (PCA), we apply K-Means clustering to group observations with similar characteristics.

These clusters may represent different market conditions such as:

- Bull Markets
- Bear Markets
- Sideways Markets
- Transitional Periods

Unlike supervised learning, no regime labels are provided to the model. Instead, the algorithm discovers patterns directly from the data.

The resulting clusters will serve as the foundation for the market regime detection system developed in this project.

Clustering is performed on the PCA-transformed dataset rather than the original engineered features.

The PCA components provide a compact representation of market behavior by combining information from returns, volatility, and momentum while reducing redundancy between variables.

Using the PCA feature space can improve cluster separation and facilitate the identification of meaningful market regimes.

-----

## Load PCA Dataset

In this section, we load the dataset generated during the PCA stage.

This dataset contains the principal components that will be used as inputs for the clustering algorithm.

The objective is to verify that the PCA transformation was successfully saved and can be loaded correctly for the market regime detection process.

In [1]:
# Import libraries
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import os 
import sys

# Ruta absoluta a la raíz del proyecto
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from config import *

In [5]:
# Load PCA DataSet
if len(ASSETS) != 1:
    raise ValueError(
        "This notebook is designed for single-asset analysis."
    )

asset_name = (
    ASSETS[0]
    .lower()
    .replace("-", "_")
)

pca_path =(
    f"{PROCESSED_DATA_PATH}/"
    f"{asset_name}_pca.csv"
)

df = pd.read_csv(pca_path)

df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

------

## Dataset Verification

Before applying clustering algorithms, we verify the structure of the PCA-transformed dataset.

The dataset should contain the principal components generated during the dimensionality reduction stage and will serve as the feature space used to identify potential market regimes.

In [6]:
# Vrification of the df
df.info()
df.describe()

<class 'pandas.DataFrame'>
DatetimeIndex: 3065 entries, 2018-01-31 to 2026-06-22
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PC1     3065 non-null   float64
 1   PC2     3065 non-null   float64
 2   PC3     3065 non-null   float64
dtypes: float64(3)
memory usage: 95.8 KB


,PC1,PC2,PC3
count,3.065000e+03,3.065000e+03,3.065000e+03
mean,-1.159124e-17,3.709195e-17,-2.318247e-17
std,1.128224e+00,1.001394e+00,8.516445e-01
min,-1.054665e+01,-1.706852e+00,-5.187152e+00
25%,-5.492385e-01,-6.763457e-01,-4.641618e-01
50%,-3.282028e-02,-2.004149e-01,-4.235113e-02
75%,5.477539e-01,4.728205e-01,4.116788e-01
max,6.087710e+00,4.982666e+00,5.821992e+00


### Interpretation

The PCA-transformed dataset contains 3,065 observations represented by three principal components (PC1, PC2, and PC3), providing a compact representation of market behavior suitable for clustering analysis.

No missing values were detected, and all principal components are stored as numerical variables, satisfying the requirements of the K-Means algorithm.

The principal components are centered around zero, which is expected after feature standardization and PCA transformation. Additionally, the decreasing standard deviations across components reflect the variance hierarchy identified during the PCA stage, where PC1 captures the largest proportion of information and PC3 captures the smallest.

Overall, the dataset appears clean, consistent, and ready for unsupervised learning. The principal component space will be used to identify potential market regimes through clustering techniques.

-----